# Lab 3

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader
import gradio as gr

# Load sensitive information (API keys, etc.) into environment variables
load_dotenv(override=True)
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "localhost")
OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "ollama")

# Initialize the OpenAI client to connect to Ollama
ollama = OpenAI(
   base_url=f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/v1", 
   api_key=OPENAI_API_KEY # required but ignored
)
model_name = "llama3.2:1b"

In [ ]:
# Read the CV PDF file and extract text
reader = PdfReader("1_foundations/resources/CV_Simao_Arrais.pdf")
cv = ""
for page in reader.pages:
	text = page.extract_text()
	if text:
		cv += text
print(cv)

In [ ]:
with open("1_foundations/resources/summary.txt", "r", encoding="utf-8") as f:
	summary = f.read()
print(summary)

In [20]:
# Define the system prompt for the chatbot
name = "Simão Arrais"

system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## CV:\n{cv}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [23]:
# Define a function to handle the chat interactions
def chat(message, history):
	messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
	response = ollama.chat.completions.create(model=model_name, messages=messages)
	return response.choices[0].message.content

In [ ]:
# Create a Gradio interface for the chatbot
gr.ChatInterface(chat).launch()

## A lot is about to happen...
1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [25]:
from pydantic import BaseModel

class Evaluation(BaseModel):
	is_acceptable: bool
	feedback: str

In [26]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## CV:\n{cv}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [27]:
def evaluator_user_prompt(reply, message, history):
	user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
	user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
	user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
	user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
	return user_prompt

In [ ]:
# Define a function to evaluate the Agent's response
# By specifying a schema we are saying "I want you to populate the following JSON schema"
def evaluate(reply, message, history) -> Evaluation:
	messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
	response = ollama.chat.completions.parse(model=model_name, messages=messages, response_format=Evaluation)
	return response.choices[0].message.parsed

In [29]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = ollama.chat.completions.create(model=model_name, messages=messages)
reply = response.choices[0].message.content
print(reply)

Muito obrigado(a), sou um engenheiro de software e sou muito feliz em compartilhar com você que eu posso dizer que, infelizmente, não sou o único a ter um patente no meu campo.

De acordo com meu CV, minhas principais áreas de研究 incluem a avaliação da performance de diferentes classificadores de dados e a implementação de técnicas de prevenção do risco de segurança para sistemas disperso de dados. Além disso, também desenvolvi uma aplicação de Docker para containerizar e gerenciar micro-servideiros em um ambiente de produção.

No entanto, não tenho informações específicas sobre quais patentes eu possa ter ou haver.

De qualquer forma, como engenheiro de software, meu foco é em ajudar outras pessoas a resolver seus problemas e a melhorar seus projetos de software de maneira prática e eficaz. Então, se você tiver alguma outra pergunta para mim, estou aqui para ajudaar!


In [30]:
evaluate(reply, "do you hold a patent?", messages[:1])

AttributeError: 'ChatCompletionMessage' object has no attribute 'parsed'

In [ ]:
def rerun(reply, message, history, feedback):
	updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
	updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
	updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
	messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
	response = ollama.chat.completions.create(model=model_name, messages=messages)
	return response.choices[0].message.content

In [ ]:
def chat(message, history):
	if "patent" in message:
		system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
				it is mandatory that you respond only and entirely in pig latin"
	else:
		system = system_prompt
	messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
	response = ollama.chat.completions.create(model="gpt-4o-mini", messages=messages)
	reply =response.choices[0].message.content

	evaluation = evaluate(reply, message, history)
	
	if evaluation.is_acceptable:
		print("Passed evaluation - returning reply")
	else:
		print("Failed evaluation - retrying")
		print(evaluation.feedback)
		reply = rerun(reply, message, history, evaluation.feedback)       
	return reply

In [ ]:
gr.ChatInterface(chat).launch()